# Publicador de eventos de piezometría (tiempo real simulado)

Equivalente en Colab de `publicador/publicar_eventos.py`. Reproduce por Pub/Sub las lecturas de los últimos 7 días del lote (gold), con las fechas corridas +7 días, como si fueran lecturas nuevas.

**Es una simulación declarada**: no hay sensores conectados, las fechas están desplazadas y se inyectan 25 mensajes con fallas sintéticas (10 entregas duplicadas, 5 instrumentos inexistentes, 5 NPA nulos y 5 NPA no numéricos).

Antes de ejecutar: haber corrido `./deploy.sh lote` y creado el tema y la suscripción (`./deploy.sh flujo` lo hace; para usar este notebook en lugar del script, ejecute solo la parte de Pub/Sub). Ejecute las celdas en orden y **una sola vez**: la celda 4 se detiene si la tabla cruda ya tiene mensajes.

Edite `PROYECTO` (y `DATASET` si lo cambió) en la celda 1.

In [ ]:
# CELDA 1 · Configuración y autenticación (se abrirá una ventana para elegir la cuenta de Google que usa en GCP)
!pip -q install google-cloud-pubsub google-cloud-bigquery
from google.colab import auth
auth.authenticate_user()
import os
PROYECTO = "ESCRIBA-AQUI-SU-PROYECTO"
DATASET  = "piezometria_demo"
REGION   = "southamerica-east1"
TEMA     = "lecturas-piezo"
assert PROYECTO != "ESCRIBA-AQUI-SU-PROYECTO", "Edite PROYECTO antes de continuar"
os.environ["GOOGLE_CLOUD_QUOTA_PROJECT"] = PROYECTO

In [ ]:
# CELDA 2 · Generación de eventos (función probada fuera de Colab)
import json, random, uuid, datetime as dt

def construir_eventos(base, desplazamiento_dias=7, semilla=42, n_dup=10, n_sin_cat=5, n_nulo=5, n_no_num=5):
    """base: lista de (instrumento_cod, fecha:date, npa_msnm). Devuelve (mensajes, resumen).
    Cada mensaje es un dict sin ts_generado (se estampa al publicar)."""
    random.seed(semilla)
    ev = []
    for ins, f, npa in base:
        ev.append({"evento_id": str(uuid.uuid4()), "instrumento_cod": ins,
                   "fecha": (f + dt.timedelta(days=desplazamiento_dias)).isoformat(), "npa_msnm": float(npa)})
    idx = random.sample(range(len(ev)), n_dup)
    dups = [dict(ev[i]) for i in idx]
    sin_cat = [dict(ev[i], evento_id=str(uuid.uuid4()), instrumento_cod="INS-999") for i in random.sample(range(len(ev)), n_sin_cat)]
    nulos = [dict(ev[i], evento_id=str(uuid.uuid4()), npa_msnm=None) for i in random.sample(range(len(ev)), n_nulo)]
    no_num = [dict(ev[i], evento_id=str(uuid.uuid4()), npa_msnm="N/D") for i in random.sample(range(len(ev)), n_no_num)]
    todo = ev + dups + sin_cat + nulos + no_num
    random.shuffle(todo)
    resumen = {"base": len(ev), "duplicados_entrega": n_dup, "instrumento_inexistente": n_sin_cat,
               "npa_nulo": n_nulo, "npa_no_numerico": n_no_num, "total_mensajes": len(todo)}
    return todo, resumen


In [ ]:
# CELDA 3 · Leer las lecturas base desde BigQuery (últimos 7 días de gold) y construir los mensajes
from google.cloud import bigquery
bq = bigquery.Client(project=PROYECTO, location=REGION)
filas = bq.query(f"""
    SELECT instrumento_cod, fecha, npa_msnm
    FROM `{PROYECTO}.{DATASET}.g_lecturas`
    WHERE fecha >= DATE_SUB((SELECT MAX(fecha) FROM `{PROYECTO}.{DATASET}.g_lecturas`), INTERVAL 6 DAY)
    ORDER BY fecha, instrumento_cod
""").result()
base = [(f.instrumento_cod, f.fecha, f.npa_msnm) for f in filas]
mensajes, resumen = construir_eventos(base)
print("Lecturas base leídas de gold:", len(base))
for k, v in resumen.items():
    print(f"  {k}: {v}")
print("Ejemplo de mensaje:", mensajes[0])

In [ ]:
# CELDA 4 · Publicar, medir latencia y verificar la llegada a BigQuery
import json, time, uuid, statistics, datetime
from google.cloud import pubsub_v1

PERMITIR_REPETIR = False   # dejar en False: esta corrida se hace una sola vez
TABLA_RAW = f"`{PROYECTO}.{DATASET}.b_eventos_raw`"
previos = list(bq.query(f"SELECT COUNT(*) AS n FROM {TABLA_RAW}").result())[0].n
if previos > 0 and not PERMITIR_REPETIR:
    raise SystemExit(f"La tabla cruda ya tiene {previos} mensajes. No se publica de nuevo (PERMITIR_REPETIR=False).")

RUN_ID = "corrida-" + time.strftime("%Y%m%d-%H%M%S", time.gmtime())
publicador = pubsub_v1.PublisherClient()
ruta_tema = publicador.topic_path(PROYECTO, TEMA)
total = len(mensajes)
acks_ms, futuros = [], []

def marcar(inicio):
    def cb(_):
        acks_ms.append((time.perf_counter() - inicio) * 1000)
    return cb

print(f"RUN_ID {RUN_ID} · publicando {total} mensajes en {ruta_tema}")
t0 = time.perf_counter()
for i, m in enumerate(mensajes, 1):
    carga = dict(m, run_id=RUN_ID, ts_generado=datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="milliseconds"))
    fut = publicador.publish(ruta_tema, json.dumps(carga).encode("utf-8"), run_id=RUN_ID)
    fut.add_done_callback(marcar(time.perf_counter()))
    futuros.append(fut)
    if i % 100 == 0 or i == total:
        print(f"  enviados {i}/{total}")
for f in futuros:
    f.result(timeout=120)
t_publicado = time.perf_counter() - t0
print(f"Todos los mensajes confirmados por Pub/Sub en {t_publicado:.2f} s")

# Espera a que todos los mensajes sean consultables en BigQuery (resolución aproximada de 2 s)
consulta = f"SELECT COUNT(*) AS n FROM {TABLA_RAW} WHERE JSON_VALUE(data, '$.run_id') = '{RUN_ID}'"
cfg = bigquery.QueryJobConfig(use_query_cache=False)
recibidos, t_visible = 0, None
while time.perf_counter() - t0 < 300:
    recibidos = list(bq.query(consulta, job_config=cfg).result())[0].n
    print(f"  en BigQuery: {recibidos}/{total}  (t = {time.perf_counter() - t0:.1f} s)")
    if recibidos >= total:
        t_visible = time.perf_counter() - t0
        break
    time.sleep(2)

print()
print("=" * 60)
print("RESUMEN DE LA CORRIDA")
print("=" * 60)
print(f"RUN_ID                         : {RUN_ID}")
print(f"Mensajes enviados              : {total}")
print(f"Mensajes recibidos en BigQuery : {recibidos}")
print(f"Diferencia                     : {total - recibidos}")
if acks_ms:
    q = statistics.quantiles(acks_ms, n=20)
    print(f"Confirmación de Pub/Sub (ms)   : mediana {statistics.median(acks_ms):.0f} · p95 {q[18]:.0f} · máximo {max(acks_ms):.0f}")
print(f"Tiempo hasta publicar todo     : {t_publicado:.2f} s")
print("Tiempo hasta que todo es consultable:", f"{t_visible:.1f} s (desde el primer envío)" if t_visible else "no se completó en 300 s")
print(f"Expectativa en silver: {resumen['duplicados_entrega']} duplicados, {resumen['instrumento_inexistente']} sin catálogo, {resumen['npa_nulo']} NPA nulos, {resumen['npa_no_numerico']} NPA no numéricos; en gold del flujo: {resumen['base']} lecturas")